In [2]:
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
from google import genai
from banking77_llm_vs_ml.core import load_data, make_schema, classify, PROMPT_V1

client = genai.Client()
MODEL = "gemini-3.5-flash-lite"

train_df, val_df, test = load_data()
categories = sorted(train_df["category"].unique()) + ["out_of_scope"]
schema = make_schema(categories)

# 3 messages per category from validation, the same every run (seed 42)
sample = val_df.groupby("category").sample(3, random_state=42).reset_index(drop=True)
print(len(sample), "messages in the eval sample")

231 messages in the eval sample


In [3]:
import time
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

p1 = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000, C=10))
p1.fit(train_df["text"], train_df["category"])

start = time.perf_counter()
sample["p1_pred"] = p1.predict(sample["text"])
p1_latency = (time.perf_counter() - start) / len(sample)
print(f"P1 latency per message: {p1_latency * 1000:.2f} ms")

P1 latency per message: 0.02 ms


In [4]:
from pathlib import Path
from google.genai import errors

RESULTS = Path("results/v1_flash_lite.csv")
RESULTS.parent.mkdir(exist_ok=True)
DELAY = 60 / 15 + 0.5   # seconds between calls, to stay under 15 requests/minute

# Resume: load anything already done, so a crash never wastes quota
rows = pd.read_csv(RESULTS).to_dict("records") if RESULTS.exists() else []
done = {row["text"] for row in rows}

for i, text in enumerate(sample["text"]):
    if text in done:
        continue

    for attempt in range(3):
        try:
            result = classify(client, MODEL, text, PROMPT_V1, schema)
            break
        except errors.APIError as e:
            if e.code == 429 and attempt < 2:
                wait = 30 * (attempt + 1)
                print(f"Rate limited, waiting {wait}s...")
                time.sleep(wait)
            else:
                result = {"pred": None, "error": str(e)}
                break

    rows.append({"text": text, **result})
    pd.DataFrame(rows).to_csv(RESULTS, index=False)   # save after every message
    print(f"{i + 1}/{len(sample)}", end="\r")
    time.sleep(DELAY)

print("\nDone:", len(rows), "results saved")

231/231
Done: 231 results saved


In [5]:
from sklearn.metrics import accuracy_score, f1_score

res = sample.merge(pd.read_csv(RESULTS), on="text")
res["pred"] = res["pred"].fillna("FAILED")

def scores(name, preds, latency_ms):
    return {
        "model": name,
        "accuracy": round(accuracy_score(res["category"], preds), 3),
        "macro_f1": round(f1_score(res["category"], preds, average="macro", zero_division=0), 3),
        "latency_ms": round(latency_ms, 1),
    }

summary = pd.DataFrame([
    scores("P1: TF-IDF + LogReg", res["p1_pred"], p1_latency * 1000),
    scores(f"LLM: {MODEL}, prompt v1", res["pred"], res["latency_s"].mean() * 1000),
])
summary

,model,accuracy,macro_f1,latency_ms
0,P1: TF-IDF + LogReg,0.913,0.911,0.0
1,"LLM: gemini-3.5-flash-lite, prompt v1",0.736,0.714,1476.8


In [6]:
print("Failed / unparsed:", (res["pred"] == "FAILED").sum())
print("Predicted out_of_scope:", (res["pred"] == "out_of_scope").sum())
print("Total tokens in / out:", res["tokens_in"].sum(), "/", res["tokens_out"].sum())
print("Latency p50 / p95 (s):", res["latency_s"].quantile(0.5), "/", res["latency_s"].quantile(0.95))

Failed / unparsed: 0
Predicted out_of_scope: 2
Total tokens in / out: 12301 / 3401
Latency p50 / p95 (s): 1.01 / 3.335


In [7]:
wrong = res[res["pred"] != res["category"]]
print("LLM wrong:", len(wrong), "of", len(res))

wrong.groupby(["category", "pred"]).size().sort_values(ascending=False).head(15)

LLM wrong: 61 of 231


category                                          pred                       
get_physical_card                                 change_pin                     3
top_up_by_bank_transfer_charge                    transfer_fee_charged           3
card_about_to_expire                              country_support                2
beneficiary_not_allowed                           transfer_into_account          2
transfer_not_received_by_recipient                transfer_timing                2
receiving_money                                   transfer_into_account          2
pending_top_up                                    top_up_failed                  2
card_payment_wrong_exchange_rate                  exchange_rate                  2
apple_pay_or_google_pay                           top_up_failed                  2
order_physical_card                               get_physical_card              2
compromised_card                                  card_payment_not_recognised    1
cash_with

In [8]:
llm_only_wrong = res[(res["p1_pred"] == res["category"]) & (res["pred"] != res["category"])]
print(len(llm_only_wrong), "messages where only the LLM failed")

pd.set_option("display.max_colwidth", None)
llm_only_wrong[["text", "category", "pred"]].head(15)

49 messages where only the LLM failed


,text,category,pred
9,"Help me please, My Apple Pay American Express top up does not work.",apple_pay_or_google_pay,top_up_failed
18,Are bank transfers immediate?,balance_not_updated_after_bank_transfer,transfer_timing
24,"I am getting an error stating my transfer is not possible while transferring money. Let me know the issue as its the first time i am getting error, earlier it used to work.",beneficiary_not_allowed,declined_transfer
25,I need to transfer to my account and cant.,beneficiary_not_allowed,transfer_into_account
31,want to get a new card in china,card_about_to_expire,country_support
32,Am I able to get this shipped to China?,card_about_to_expire,country_support
38,Can you track my card for me?,card_arrival,card_delivery_estimate
53,I don't understand where this charge came from.,card_payment_not_recognised,extra_charge_on_statement
54,Are you sure this is the interbank exchange rate? This rate is really bad.,card_payment_wrong_exchange_rate,exchange_rate
56,"Hello, I believe there has been a mistake made on the exchange rate for my card. Could you please check the official interbank exchange rate for me?",card_payment_wrong_exchange_rate,exchange_rate


In [9]:
p1_only_wrong = res[(res["p1_pred"] != res["category"]) & (res["pred"] == res["category"])]
print(len(p1_only_wrong), "messages where only P1 failed")
p1_only_wrong[["text", "category", "p1_pred"]]

8 messages where only P1 failed


,text,category,p1_pred
17,How can I Use thereto-top option?,automatic_top_up,top_up_by_cash_or_cheque
57,Why did the ATM take my card and keep it?,card_swallowed,declined_cash_withdrawal
61,I made a cash withdrawl and there is a charge. What is this?,cash_withdrawal_charge,cash_withdrawal_not_recognised
76,Is this card available to Americans?,country_support,card_arrival
128,"I thought I left my card at a restaurant, but they claim not to have it and now I don't have any idea where it is. I'm so worried that someone might be using it fraudulently. Is there any way to look into this?",lost_or_stolen_card,card_payment_not_recognised
200,where has my transferred money gone?,transfer_not_received_by_recipient,balance_not_updated_after_bank_transfer
218,I cannot get my virtual card to function.,virtual_card_not_working,getting_virtual_card
223,Why does my identity need to be verified?,why_verify_identity,unable_to_verify_identity
